In [2]:
!pip install -q kaggle open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.5 MB/s eta 0:00:00


In [1]:
import os
import numpy as np

os.environ["KAGGLE_USERNAME"] = "sairaudhrankomuroju"
os.environ["KAGGLE_KEY"]      = "KGAT_9e2cb5e08a1c89b824635e13a1e05105"

# Verify access
!kaggle datasets files hbenallal/semantickitti | head -15

Next Page Token = CfDJ8OasvsPNS7VMiSwiBm2YVpM8hZtgFfHA3PmQ4QiaYBnK7aZiNgqbNhUvv4z3AOwTjsTpGk32wrkyNivBJ425usj5PEeooEHI-H1ejHBrgRqdh0ZnR0RR4uIKhHRTS9tF3BymRKKbjSGsTVGkraKl8E1_Bm9VABaxo86YNujJ9vWC6A
name                                         size  creationDate                
----------------------------------------  -------  --------------------------  
dataset/sequences/00/velodyne/000000.bin  1994688  2026-07-17 22:55:33.212000  
dataset/sequences/00/velodyne/000001.bin  1993680  2026-07-17 22:53:48.187000  
dataset/sequences/00/velodyne/000002.bin  1991648  2026-07-17 22:53:16.116000  
dataset/sequences/00/velodyne/000003.bin  1986672  2026-07-17 22:54:31.798000  
dataset/sequences/00/velodyne/000004.bin  1983504  2026-07-17 22:56:23.601000  
dataset/sequences/00/velodyne/000005.bin  1982784  2026-07-17 22:56:20.817000  
dataset/sequences/00/velodyne/000006.bin  1973968  2026-07-17 22:57:15.395000  
dataset/sequences/00/velodyne/000007.bin  1964240  2026-07-17 22:54:36.229000  
dat

In [3]:
import os

SAMPLE_DIR = "/content/semantic_kitti_sample"
os.makedirs(SAMPLE_DIR, exist_ok=True)

for i in range(10):
    fid = f"{i:06d}"

    # .bin file
    if not os.path.exists(f"{SAMPLE_DIR}/{fid}.bin"):
        os.system(
            f"kaggle datasets download hbenallal/semantickitti "
            f"-f dataset/sequences/00/velodyne/{fid}.bin "
            f"-p {SAMPLE_DIR} --unzip 2>/dev/null"
        )
        src = (f"{SAMPLE_DIR}/dataset/sequences/"
               f"00/velodyne/{fid}.bin")
        if os.path.exists(src):
            os.rename(src, f"{SAMPLE_DIR}/{fid}.bin")

    # .label file
    if not os.path.exists(f"{SAMPLE_DIR}/{fid}.label"):
        os.system(
            f"kaggle datasets download hbenallal/semantickitti "
            f"-f dataset/sequences/00/labels/{fid}.label "
            f"-p {SAMPLE_DIR} --unzip 2>/dev/null"
        )
        src = (f"{SAMPLE_DIR}/dataset/sequences/"
               f"00/labels/{fid}.label")
        if os.path.exists(src):
            os.rename(src, f"{SAMPLE_DIR}/{fid}.label")

# Verify
print("Downloaded files:")
for f in sorted(os.listdir(SAMPLE_DIR)):
    if f.endswith(".bin") or f.endswith(".label"):
        size = os.path.getsize(
            os.path.join(SAMPLE_DIR, f)
        ) / 1024
        print(f"  {f}  {size:.1f} KB")

Downloaded files:
  000000.bin  1947.9 KB
  000001.bin  1947.0 KB
  000002.bin  1945.0 KB
  000003.bin  1940.1 KB
  000004.bin  1937.0 KB
  000005.bin  1936.3 KB
  000006.bin  1927.7 KB
  000007.bin  1918.2 KB
  000008.bin  1908.2 KB
  000009.bin  1896.2 KB


In [4]:
import numpy as np

fid    = "000000"
points = np.fromfile(
    f"{SAMPLE_DIR}/{fid}.bin",
    dtype=np.float32
).reshape(-1, 4)

dist = np.sqrt(points[:,0]**2 + points[:,1]**2)

print("=" * 55)
print("DATA EXPLORATION — frame 000000")
print("=" * 55)
print(f"\n  Shape          : {points.shape}")
print(f"  Points         : {len(points):,}")
print(f"\n  X range        : {points[:,0].min():.2f} "
      f"to {points[:,0].max():.2f} m")
print(f"  Y range        : {points[:,1].min():.2f} "
      f"to {points[:,1].max():.2f} m")
print(f"  Z range        : {points[:,2].min():.2f} "
      f"to {points[:,2].max():.2f} m")
print(f"  Intensity      : {points[:,3].min():.2f} "
      f"to {points[:,3].max():.2f}")
print(f"  NaN count      : {np.isnan(points).sum()}")
print(f"\n  Horizontal dist:")
print(f"    Min          : {dist.min():.2f} m")
print(f"    Max          : {dist.max():.2f} m")

print(f"\n  Distance percentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"    {p:3d}%       : {np.percentile(dist, p):.2f} m")

# Band distribution
for label, lo, hi in [
    ("0–10m",   0,  10),
    ("10–30m",  10, 30),
    ("30–100m", 30, 100),
]:
    mask = (dist >= lo) & (dist < hi)
    print(f"\n  {label}: {mask.sum():,} points "
          f"({mask.mean()*100:.1f}%)")

DATA EXPLORATION — frame 000000

  Shape          : (124668, 4)
  Points         : 124,668

  X range        : -78.09 to 77.97 m
  Y range        : -55.72 to 44.88 m
  Z range        : -11.56 to 2.83 m
  Intensity      : 0.00 to 0.99
  NaN count      : 0

  Horizontal dist:
    Min          : 1.25 m
    Max          : 79.74 m

  Distance percentiles:
     25%       : 6.59 m
     50%       : 9.99 m
     75%       : 15.81 m
     90%       : 26.04 m
     95%       : 37.78 m
     99%       : 56.62 m

  0–10m: 62,364 points (50.0%)

  10–30m: 52,932 points (42.5%)

  30–100m: 9,372 points (7.5%)


In [5]:
import numpy as np
import time
import os

# ── CONFIG ────────────────────────────────────────────────
SAMPLE_DIR = "/content/semantic_kitti_sample"
VOXEL_DIR  = "/content/voxel_store"
os.makedirs(VOXEL_DIR, exist_ok=True)

FRAME_IDS = [f"{i:06d}" for i in range(10)]
SIZE_MAP  = np.array([0.05, 0.15, 0.50], dtype=np.float32)

BANDS = [
    (0,  10,  0.05, 0),
    (10, 30,  0.15, 1),
    (30, 100, 0.50, 2),
]

# ── PIPELINE CLASS ────────────────────────────────────────
class VoxelPipeline:
    """
    Adaptive voxelization with pre-allocated buffers.
    Processes one LiDAR frame at a time.
    """
    def __init__(self,
                 max_points=150000,
                 max_voxels=100000):
        # Pre-allocated buffers — reused every frame
        self.levels   = np.zeros(max_points, dtype=np.int32)
        self.vox_size = np.zeros(max_points, dtype=np.float32)
        self.dist     = np.zeros(max_points, dtype=np.float32)
        self.count    = np.zeros(max_voxels, dtype=np.int32)
        self.z_sum    = np.zeros(max_voxels, dtype=np.float64)
        self.z_sum2   = np.zeros(max_voxels, dtype=np.float64)
        self.z_min    = np.zeros(max_voxels, dtype=np.float64)
        self.z_max    = np.zeros(max_voxels, dtype=np.float64)
        self.i_sum    = np.zeros(max_voxels, dtype=np.float64)

    def process(self, points):
        N = len(points)

        # ── Step 1: assign resolution level per point ──
        dist     = self.dist[:N]
        levels   = self.levels[:N]
        vox_size = self.vox_size[:N]

        np.sqrt(points[:,0]**2 + points[:,1]**2, out=dist)

        levels[:]   = 2;  vox_size[:] = 0.50   # default: far
        m1 = dist < 30;   levels[m1]  = 1; vox_size[m1] = 0.15
        m0 = dist < 10;   levels[m0]  = 0; vox_size[m0] = 0.05

        # ── Step 2: compute voxel coordinates ─────────
        coords = np.floor(
            points[:,:3] / vox_size[:,None]
        ).astype(np.int32)

        # ── Step 3: build hash grid keys ──────────────
        # key = (level, ix, iy, iz) → unique voxel id
        keys   = np.stack(
            [levels,
             coords[:,0],
             coords[:,1],
             coords[:,2]],
            axis=1
        ).astype(np.int32)

        keys_c = np.ascontiguousarray(keys)
        keys_v = keys_c.view(
            np.dtype((np.void,
                      keys_c.dtype.itemsize * 4))
        ).ravel()

        unique_void, inverse = np.unique(
            keys_v, return_inverse=True
        )
        unique_keys = (unique_void
                       .view(np.int32)
                       .reshape(-1, 4))

        # ── Step 4: accumulate voxel features ─────────
        V     = len(unique_keys)
        z     = points[:,2].astype(np.float64)
        inten = points[:,3].astype(np.float64)

        count  = self.count[:V];  count[:]  = 0
        z_sum  = self.z_sum[:V];  z_sum[:]  = 0.0
        z_sum2 = self.z_sum2[:V]; z_sum2[:] = 0.0
        z_min  = self.z_min[:V];  z_min[:]  =  np.inf
        z_max  = self.z_max[:V];  z_max[:]  = -np.inf
        i_sum  = self.i_sum[:V];  i_sum[:]  = 0.0

        np.add.at(count,  inverse, 1)
        np.add.at(z_sum,  inverse, z)
        np.add.at(z_sum2, inverse, z**2)
        np.minimum.at(z_min, inverse, z)
        np.maximum.at(z_max, inverse, z)
        np.add.at(i_sum,  inverse, inten)

        z_mean  = z_sum  / count
        z_var   = z_sum2 / count - z_mean**2
        z_range = z_max  - z_min
        i_mean  = i_sum  / count

        # ── Step 5: voxel centers ─────────────────────
        lv = unique_keys[:,0]
        vs = SIZE_MAP[lv]
        cx = (unique_keys[:,1] + 0.5) * vs
        cy = (unique_keys[:,2] + 0.5) * vs
        cz = (unique_keys[:,3] + 0.5) * vs

        features = {
            "unique_keys":    unique_keys,
            "levels":         lv,
            "vox_size":       vs,
            "center_x":       cx,
            "center_y":       cy,
            "center_z":       cz,
            "point_count":    count.copy(),
            "height_mean":    z_mean.copy(),
            "height_min":     z_min.copy(),
            "height_max":     z_max.copy(),
            "height_var":     z_var.copy(),
            "height_range":   z_range.copy(),
            "intensity_mean": i_mean.copy(),
        }

        # inverse maps each point → its voxel index
        return features, inverse.astype(np.int32)


# ── SAVE ─────────────────────────────────────────────────
def save_voxels(features, inverse, frame_id):
    center_xyz = np.stack([
        features["center_x"],
        features["center_y"],
        features["center_z"]
    ], axis=1).astype(np.float32)

    np.savez_compressed(
        os.path.join(VOXEL_DIR, f"{frame_id}_voxels.npz"),
        keys         = features["unique_keys"].astype(np.int32),
        levels       = features["levels"].astype(np.int32),
        vox_size     = features["vox_size"].astype(np.float32),
        center_xyz   = center_xyz,
        point_count  = features["point_count"].astype(np.int32),
        height_mean  = features["height_mean"].astype(np.float32),
        height_min   = features["height_min"].astype(np.float32),
        height_max   = features["height_max"].astype(np.float32),
        height_var   = features["height_var"].astype(np.float32),
        height_range = features["height_range"].astype(np.float32),
        intensity    = features["intensity_mean"].astype(np.float32),
        inverse      = inverse,  # point → voxel mapping
    )


# ── LOAD ─────────────────────────────────────────────────
def load_voxels(frame_id):
    d = np.load(
        os.path.join(VOXEL_DIR, f"{frame_id}_voxels.npz")
    )
    return {
        "unique_keys": d["keys"],
        "levels":      d["levels"],
        "vox_size":    d["vox_size"],
        "center_x":    d["center_xyz"][:,0],
        "center_y":    d["center_xyz"][:,1],
        "center_z":    d["center_xyz"][:,2],
        "point_count": d["point_count"],
        "height_mean": d["height_mean"],
        "height_min":  d["height_min"],
        "height_max":  d["height_max"],
        "height_var":  d["height_var"],
        "height_range":d["height_range"],
        "intensity_mean": d["intensity"],
        "inverse":     d["inverse"],
    }


# ── UNIFORM BASELINE (for benchmark) ─────────────────────
def uniform_voxel_count(points, size=0.05):
    coords = np.floor(
        points[:,:3] / size
    ).astype(np.int32)
    keys   = np.ascontiguousarray(coords).view(
        np.dtype((np.void, 12))
    ).ravel()
    return len(np.unique(keys))


# ── MAIN LOOP ─────────────────────────────────────────────
print("=" * 80)
print("ADAPTIVE VOXELIZATION — 10 FRAMES")
print("=" * 80)

pipeline = VoxelPipeline(
    max_points=150000,
    max_voxels=100000
)

# Warmup call — first np.unique is always slower
pts_w = np.fromfile(
    f"{SAMPLE_DIR}/000000.bin",
    dtype=np.float32
).reshape(-1, 4)
pipeline.process(pts_w)

print(f"\n{'Frame':<12} {'Points':>8} {'Uniform':>10} "
      f"{'Adaptive':>10} {'Reduction':>10} "
      f"{'L0':>7} {'L1':>7} {'L2':>7} "
      f"{'ms':>8} {'FPS':>6}")
print("-" * 95)

all_results = []

for fid in FRAME_IDS:
    points = np.fromfile(
        f"{SAMPLE_DIR}/{fid}.bin",
        dtype=np.float32
    ).reshape(-1, 4)

    # Uniform count (benchmark only, outside timer)
    unif = uniform_voxel_count(points)

    # Pipeline — timed
    t0             = time.perf_counter()
    features, inv  = pipeline.process(points)
    elapsed        = (time.perf_counter() - t0) * 1000

    save_voxels(features, inv, fid)

    from collections import Counter
    lv_counts = Counter(features["levels"].tolist())
    adap      = len(features["unique_keys"])

    all_results.append({
        "frame":     fid,
        "points":    len(points),
        "uniform":   unif,
        "adaptive":  adap,
        "reduction": unif / adap,
        "l0":        lv_counts[0],
        "l1":        lv_counts[1],
        "l2":        lv_counts[2],
        "ms":        elapsed,
    })

    print(f"{fid:<12} {len(points):>8,} {unif:>10,} "
          f"{adap:>10,} {unif/adap:>9.2f}x "
          f"{lv_counts[0]:>7,} {lv_counts[1]:>7,} "
          f"{lv_counts[2]:>7,} {elapsed:>7.1f}ms "
          f"{1000/elapsed:>5.1f}")

def avg(k): return np.mean([r[k] for r in all_results])

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"  Avg points     : {avg('points'):,.0f}")
print(f"  Avg uniform    : {avg('uniform'):,.0f}")
print(f"  Avg adaptive   : {avg('adaptive'):,.0f}")
print(f"  Avg reduction  : {avg('reduction'):.2f}x")
print(f"  Avg time       : {avg('ms'):.1f} ms")
print(f"  Avg FPS        : {1000/avg('ms'):.1f}")
print(f"  Avg L0 (5cm)   : {avg('l0'):,.0f} "
      f"({avg('l0')/avg('adaptive')*100:.1f}%)")
print(f"  Avg L1 (15cm)  : {avg('l1'):,.0f} "
      f"({avg('l1')/avg('adaptive')*100:.1f}%)")
print(f"  Avg L2 (50cm)  : {avg('l2'):,.0f} "
      f"({avg('l2')/avg('adaptive')*100:.1f}%)")
print(f"\n  Voxel files saved to: {VOXEL_DIR}/")

ADAPTIVE VOXELIZATION — 10 FRAMES

Frame          Points    Uniform   Adaptive  Reduction      L0      L1      L2       ms    FPS
-----------------------------------------------------------------------------------------------
000000        124,668     91,767     62,158      1.48x  33,922  23,934   4,302    75.5ms  13.2
000001        124,605     91,495     62,256      1.47x  34,389  23,592   4,275   215.9ms   4.6
000002        124,478     91,177     62,134      1.47x  34,635  23,373   4,126   253.7ms   3.9
000003        124,167     90,572     61,926      1.46x  35,151  22,772   4,003    96.4ms  10.4
000004        123,969     90,316     61,899      1.46x  35,461  22,426   4,012    46.7ms  21.4
000005        123,924     89,992     61,519      1.46x  35,126  22,359   4,034    46.7ms  21.4
000006        123,373     89,469     60,909      1.47x  34,838  22,167   3,904    46.6ms  21.5
000007        122,765     89,219     60,534      1.47x  34,671  21,969   3,894    45.6ms  21.9
000008        

In [6]:
import numpy as np
import time
import os

MAP_DIR = "/content/maps_2d"
os.makedirs(MAP_DIR, exist_ok=True)

# Feature channel indices
CH = {
    "height_max":   0,
    "height_min":   1,
    "height_range": 2,
    "height_mean":  3,
    "point_count":  4,
    "voxel_count":  5,
    "intensity":    6,
    "is_occupied":  7,
    "pts_per_m2":   8,
}
N_CH = len(CH)

SIZE_MAP  = np.array([0.05, 0.15, 0.50], dtype=np.float32)
FRAME_IDS = [f"{i:06d}" for i in range(10)]

# ── PROJECTION ────────────────────────────────────────────
def project_level(features, level):
    mask = features["levels"] == level
    if mask.sum() == 0:
        return None, None

    vs     = float(SIZE_MAP[level])
    cx     = features["center_x"][mask]
    cy     = features["center_y"][mask]

    ix     = np.floor(cx / vs).astype(np.int32)
    iy     = np.floor(cy / vs).astype(np.int32)
    ix_min = ix.min();  iy_min = iy.min()
    ix_rel = ix - ix_min
    iy_rel = iy - iy_min
    H      = int(ix_rel.max()) + 1
    W      = int(iy_rel.max()) + 1
    flat   = ix_rel * W + iy_rel
    C      = H * W

    h_max  = features["height_max"][mask].astype(np.float32)
    h_min  = features["height_min"][mask].astype(np.float32)
    h_mean = features["height_mean"][mask].astype(np.float32)
    pcount = features["point_count"][mask].astype(np.float32)
    inten  = features["intensity_mean"][mask].astype(np.float32)

    g_hmax    = np.full(C, -np.inf, dtype=np.float32)
    g_hmin    = np.full(C,  np.inf, dtype=np.float32)
    g_pcount  = np.zeros(C, dtype=np.float32)
    g_vcount  = np.zeros(C, dtype=np.float32)
    g_hmean_w = np.zeros(C, dtype=np.float32)
    g_inten_w = np.zeros(C, dtype=np.float32)

    np.maximum.at(g_hmax,    flat, h_max)
    np.minimum.at(g_hmin,    flat, h_min)
    np.add.at(g_pcount,      flat, pcount)
    np.add.at(g_vcount,      flat, 1)
    np.add.at(g_hmean_w,     flat, h_mean * pcount)
    np.add.at(g_inten_w,     flat, inten  * pcount)

    occ      = g_vcount > 0
    g_hmean  = np.zeros(C, dtype=np.float32)
    g_inten  = np.zeros(C, dtype=np.float32)
    g_hrange = np.zeros(C, dtype=np.float32)

    g_hmean[occ]  = g_hmean_w[occ] / g_pcount[occ]
    g_inten[occ]  = g_inten_w[occ] / g_pcount[occ]
    g_hrange[occ] = g_hmax[occ] - g_hmin[occ]
    g_hmax[~occ]  = 0.0
    g_hmin[~occ]  = 0.0

    grid = np.zeros((H, W, N_CH), dtype=np.float32)
    grid[:,:,CH["height_max"]]   = g_hmax.reshape(H, W)
    grid[:,:,CH["height_min"]]   = g_hmin.reshape(H, W)
    grid[:,:,CH["height_range"]] = g_hrange.reshape(H, W)
    grid[:,:,CH["height_mean"]]  = g_hmean.reshape(H, W)
    grid[:,:,CH["point_count"]]  = g_pcount.reshape(H, W)
    grid[:,:,CH["voxel_count"]]  = g_vcount.reshape(H, W)
    grid[:,:,CH["intensity"]]    = g_inten.reshape(H, W)
    grid[:,:,CH["is_occupied"]]  = occ.reshape(H,W).astype(np.float32)
    grid[:,:,CH["pts_per_m2"]]   = (g_pcount / vs**2).reshape(H, W)

    meta = {
        "H":        H,
        "W":        W,
        "vox_size": vs,
        "ix_min":   int(ix_min),
        "iy_min":   int(iy_min),
        "occupied": int(occ.sum()),
    }
    return grid, meta


def project_2d(features):
    grids = {};  metas = {}
    for lv in [0, 1, 2]:
        grid, meta = project_level(features, lv)
        if grid is not None:
            grids[lv] = grid
            metas[lv] = meta
    return grids, metas


def save_map(grids, metas, frame_id):
    meta_arr = np.array([
        [metas[lv]["H"],
         metas[lv]["W"],
         metas[lv]["ix_min"],
         metas[lv]["iy_min"]]
        for lv in [0, 1, 2]
    ], dtype=np.int32)

    np.savez_compressed(
        os.path.join(MAP_DIR, f"{frame_id}_map2d.npz"),
        L0_5cm     = grids[0],
        L1_15cm    = grids[1],
        L2_50cm    = grids[2],
        meta_hwoff = meta_arr,
    )


def load_map(frame_id):
    d        = np.load(
        os.path.join(MAP_DIR, f"{frame_id}_map2d.npz")
    )
    meta_arr = d["meta_hwoff"]
    grids    = {
        0: d["L0_5cm"],
        1: d["L1_15cm"],
        2: d["L2_50cm"],
    }
    metas = {}
    for i, lv in enumerate([0, 1, 2]):
        H, W, ix_min, iy_min = meta_arr[i]
        metas[lv] = {
            "H":        int(H),
            "W":        int(W),
            "ix_min":   int(ix_min),
            "iy_min":   int(iy_min),
            "vox_size": float(SIZE_MAP[lv]),
        }
    return grids, metas


# ── MAIN LOOP ─────────────────────────────────────────────
print("=" * 70)
print("2.5D MAP PROJECTION — 10 FRAMES")
print("=" * 70)

print(f"\n{'Frame':<12} {'L0 grid':>14} {'L1 grid':>14} "
      f"{'L2 grid':>14} {'ms':>8} {'FPS':>6}")
print("-" * 75)

all_proj_ms = []

for fid in FRAME_IDS:
    features = load_voxels(fid)

    t0           = time.perf_counter()
    grids, metas = project_2d(features)
    elapsed      = (time.perf_counter() - t0) * 1000
    all_proj_ms.append(elapsed)

    save_map(grids, metas, fid)

    l0 = metas[0];  l1 = metas[1];  l2 = metas[2]
    print(f"{fid:<12} "
          f"{l0['H']}×{l0['W']:>5} "
          f"{l1['H']}×{l1['W']:>5} "
          f"{l2['H']}×{l2['W']:>5} "
          f"{elapsed:>8.1f}ms "
          f"{1000/elapsed:>5.1f}")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"  Avg projection time : {np.mean(all_proj_ms):.1f} ms")
print(f"  Avg FPS             : {1000/np.mean(all_proj_ms):.1f}")
print(f"\n  Occupancy (frame 0):")
grids_0, metas_0 = load_map("000000")
for lv in [0, 1, 2]:
    g   = grids_0[lv]
    occ = g[:,:,CH["is_occupied"]].astype(bool).sum()
    tot = metas_0[lv]["H"] * metas_0[lv]["W"]
    print(f"    L{lv}: {occ:,} / {tot:,} cells "
          f"({occ/tot*100:.1f}% occupied)")
print(f"\n  Maps saved to: {MAP_DIR}/")

2.5D MAP PROJECTION — 10 FRAMES

Frame               L0 grid        L1 grid        L2 grid       ms    FPS
---------------------------------------------------------------------------
000000       397×  399 398×  342 313×  202     43.4ms  23.0
000001       399×  400 399×  332 319×  213    152.5ms   6.6
000002       400×  400 400×  324 317×  196     99.1ms  10.1
000003       399×  400 398×  317 318×  181     44.8ms  22.3
000004       398×  396 398×  309 317×  176    101.3ms   9.9
000005       396×  396 398×  302 313×  172     40.2ms  24.9
000006       391×  393 399×  296 313×  171     25.8ms  38.7
000007       395×  389 399×  286 317×  170     29.8ms  33.6
000008       400×  388 400×  279 316×  169     30.1ms  33.2
000009       398×  390 400×  274 315×  169     25.7ms  38.8

SUMMARY
  Avg projection time : 59.3 ms
  Avg FPS             : 16.9

  Occupancy (frame 0):
    L0: 25,455 / 158,403 cells (16.1% occupied)
    L1: 13,358 / 136,116 cells (9.8% occupied)
    L2: 2,512 / 63,226 cells

In [8]:
import numpy as np
import time
import os
import json

SPARSE_DIR = "/content/sparse5_store"
os.makedirs(SPARSE_DIR, exist_ok=True)

FRAME_IDS  = [f"{i:06d}" for i in range(10)]
SIZE_MAP   = np.array([0.05, 0.15, 0.50], dtype=np.float32)
N_CLASSES  = 5
IGNORE     = 0

CLASS_NAMES = {
    0: "ignore",
    1: "drivable",
    2: "non_drivable_terrain",
    3: "static_obstacle",
    4: "dynamic_object",
}

CH = {
    "height_max":   0, "height_min":   1,
    "height_range": 2, "height_mean":  3,
    "point_count":  4, "voxel_count":  5,
    "intensity":    6, "is_occupied":  7,
    "pts_per_m2":   8,
}

# ── LABEL LOOKUP TABLE ────────────────────────────────────
RAW_LABELS = {
    0:  "unlabeled",     1:  "outlier",
    10: "car",           11: "bicycle",
    13: "bus",           15: "motorcycle",
    16: "on-rails",      18: "truck",
    20: "other-vehicle",
    30: "person",        31: "bicyclist",
    32: "motorcyclist",
    40: "road",          44: "parking",
    48: "sidewalk",      49: "other-ground",
    50: "building",      51: "fence",
    52: "other-structure",
    60: "lane-marking",
    70: "vegetation",    71: "trunk",
    72: "terrain",       80: "pole",
    81: "traffic-sign",  99: "other-object",
    252: "moving-car",         253: "moving-bicyclist",
    254: "moving-person",      255: "moving-motorcyclist",
    256: "moving-on-rails",    257: "moving-bus",
    258: "moving-truck",       259: "moving-other-vehicle",
}

NAME_TO_CLASS = {
    "unlabeled": 0,    "outlier": 0,
    "other-object": 0, "other-structure": 3,
    "road": 1,         "parking": 1,
    "lane-marking": 1,
    "sidewalk": 2,     "terrain": 2,
    "other-ground": 2,
    "building": 3,     "fence": 3,
    "pole": 3,         "traffic-sign": 3,
    "vegetation": 3,   "trunk": 3,
    "car": 4,          "bicycle": 4,
    "bus": 4,          "motorcycle": 4,
    "on-rails": 4,     "truck": 4,
    "other-vehicle": 4,"person": 4,
    "bicyclist": 4,    "motorcyclist": 4,
    "moving-car": 4,          "moving-bicyclist": 4,
    "moving-person": 4,       "moving-motorcyclist": 4,
    "moving-on-rails": 4,     "moving-bus": 4,
    "moving-truck": 4,        "moving-other-vehicle": 4,
}

_MAX_ID = max(RAW_LABELS.keys()) + 1
LUT     = np.zeros(_MAX_ID, dtype=np.uint8)
for raw_id, name in RAW_LABELS.items():
    LUT[raw_id] = NAME_TO_CLASS[name]

# ── FUNCTIONS ─────────────────────────────────────────────
def remap_labels(frame_id):
    path    = os.path.join(SAMPLE_DIR, f"{frame_id}.label")
    raw     = np.fromfile(path, dtype=np.uint32)
    raw_sem = (raw & 0xFFFF).astype(np.int32)
    return LUT[np.clip(raw_sem, 0, _MAX_ID - 1)]


def assign_voxel_labels(sem_label, features):
    """Majority vote per voxel using stored inverse."""
    inverse   = features["inverse"]
    V         = len(features["unique_keys"])
    combined  = (inverse * N_CLASSES
                 + sem_label.astype(np.int32))
    flat_hist = np.bincount(
        combined, minlength=V * N_CLASSES
    ).reshape(V, N_CLASSES).astype(np.float32)

    voxel_class = np.argmax(
        flat_hist, axis=1
    ).astype(np.uint8)
    total       = flat_hist.sum(axis=1)
    voxel_conf  = (
        flat_hist[
            np.arange(V),
            voxel_class.astype(np.int32)
        ] / np.maximum(total, 1)
    ).astype(np.float32)

    return voxel_class, voxel_conf


def assign_cell_labels(features, voxel_class, metas):
    """Project voxel labels into 2D cells."""
    levels_arr = features["levels"]
    pcount_arr = features["point_count"].astype(np.float32)
    cx_all     = features["center_x"]
    cy_all     = features["center_y"]
    cell_labels = {};  cell_conf = {}

    for lv in [0, 1, 2]:
        meta   = metas[lv]
        H      = meta["H"];  W = meta["W"]
        vs     = meta["vox_size"]
        ix_min = meta["ix_min"]
        iy_min = meta["iy_min"]

        mask = levels_arr == lv
        if mask.sum() == 0:
            cell_labels[lv] = np.zeros((H,W), dtype=np.uint8)
            cell_conf[lv]   = np.zeros((H,W), dtype=np.float32)
            continue

        cx = cx_all[mask];  cy = cy_all[mask]
        vc = voxel_class[mask].astype(np.int32)
        pc = pcount_arr[mask]

        ix_rel = (np.floor(cx/vs).astype(np.int32) - ix_min)
        iy_rel = (np.floor(cy/vs).astype(np.int32) - iy_min)
        valid  = ((ix_rel>=0)&(ix_rel<H)&
                  (iy_rel>=0)&(iy_rel<W))

        ix_rel = ix_rel[valid];  iy_rel = iy_rel[valid]
        vc     = vc[valid];      pc     = pc[valid]

        flat      = ix_rel * W + iy_rel
        combined  = (flat * N_CLASSES + vc).astype(np.int64)
        flat_hist = np.bincount(
            combined,
            weights   = pc,
            minlength = H * W * N_CLASSES
        ).reshape(H*W, N_CLASSES).astype(np.float32)

        cls_flat  = np.argmax(
            flat_hist, axis=1
        ).astype(np.uint8)
        tot_flat  = flat_hist.sum(axis=1)
        occ       = tot_flat > 0
        conf_flat = np.zeros(H*W, dtype=np.float32)
        conf_flat[occ] = (
            flat_hist[occ, cls_flat[occ].astype(np.int32)]
            / tot_flat[occ]
        )
        cell_labels[lv] = cls_flat.reshape(H, W)
        cell_conf[lv]   = conf_flat.reshape(H, W)

    return cell_labels, cell_conf


def save_sparse(frame_id, grids, metas,
                cell_labels, cell_conf):
    counts = {}
    for lv in [0, 1, 2]:
        grid     = grids[lv]
        labels   = cell_labels[lv]
        conf     = cell_conf[lv]
        occupied = grid[:,:,CH["is_occupied"]].astype(bool)
        valid    = occupied & (labels != IGNORE)
        counts[lv] = int(valid.sum())

        if valid.sum() == 0:
            continue

        ix, iy = np.where(valid)
        base   = os.path.join(
            SPARSE_DIR, f"{frame_id}_l{lv}"
        )
        np.save(base + "_coords.npy",
                np.stack([ix,iy],axis=1).astype(np.int32))
        np.save(base + "_features.npy",
                grid[ix,iy,:].astype(np.float32))
        np.save(base + "_labels.npy",
                labels[ix,iy].astype(np.uint8))
        np.save(base + "_confidence.npy",
                conf[ix,iy].astype(np.float32))
    return counts


def verify_sparse(frame_id):
    errors = []
    for lv in [0, 1, 2]:
        base = os.path.join(
            SPARSE_DIR, f"{frame_id}_l{lv}"
        )
        paths = {
            k: base + f"_{k}.npy"
            for k in ["coords","features",
                      "labels","confidence"]
        }
        if not all(os.path.exists(p)
                   for p in paths.values()):
            errors.append(f"L{lv} files missing")
            continue

        coords = np.load(paths["coords"])
        feats  = np.load(paths["features"])
        labels = np.load(paths["labels"])
        conf   = np.load(paths["confidence"])
        N      = len(coords)

        if feats.shape  != (N, 9):
            errors.append(f"L{lv} feat shape {feats.shape}")
        if labels.shape != (N,):
            errors.append(f"L{lv} label shape")
        if (labels == 0).any():
            errors.append(f"L{lv} ignore labels present")
        if np.isnan(feats).any():
            errors.append(f"L{lv} NaN in features")

    return errors

# ── MAIN LOOP ─────────────────────────────────────────────
# Confirm inverse is available
test     = np.load(
    os.path.join(VOXEL_DIR, "000000_voxels.npz")
)
has_inv  = "inverse" in test.files
print(f"Inverse in voxel files : "
      f"{'✓ fast path' if has_inv else '✗ MISSING'}")
if not has_inv:
    raise RuntimeError(
        "Run Cell 5 first — inverse not saved."
    )

print(f"\n{'Frame':<10} {'L0':>7} {'L1':>7} {'L2':>7} "
      f"{'Lbl ms':>8} {'Vox ms':>8} {'Cell ms':>8} "
      f"{'Save ms':>8} {'Total ms':>10} {'FPS':>6}  Check")
print("-" * 100)

all_results  = []
class_totals = np.zeros(N_CLASSES, dtype=np.int64)
t_lbl_acc  = [];  t_vox_acc  = []
t_cell_acc = [];  t_save_acc = []

for fid in FRAME_IDS:
    t_full = time.perf_counter()

    # Load everything
    features     = load_voxels(fid)
    grids, metas = load_map(fid)

    # Remap labels
    t0     = time.perf_counter()
    sem_5  = remap_labels(fid)
    t_lbl  = (time.perf_counter() - t0) * 1000
    t_lbl_acc.append(t_lbl)

    # Voxel labels
    t0 = time.perf_counter()
    voxel_class, voxel_conf = assign_voxel_labels(
        sem_5, features
    )
    t_vox = (time.perf_counter() - t0) * 1000
    t_vox_acc.append(t_vox)

    # Cell labels
    t0 = time.perf_counter()
    cell_labels, cell_conf = assign_cell_labels(
        features, voxel_class, metas
    )
    t_cell = (time.perf_counter() - t0) * 1000
    t_cell_acc.append(t_cell)

    # Save
    t0     = time.perf_counter()
    counts = save_sparse(
        fid, grids, metas, cell_labels, cell_conf
    )
    t_save = (time.perf_counter() - t0) * 1000
    t_save_acc.append(t_save)

    elapsed = (time.perf_counter() - t_full) * 1000
    errors  = verify_sparse(fid)
    check   = "✓" if not errors else f"✗ {errors[0]}"

    # Class distribution
    lbl_path = os.path.join(
        SPARSE_DIR, f"{fid}_l1_labels.npy"
    )
    if os.path.exists(lbl_path):
        lbl = np.load(lbl_path).astype(np.int32)
        for c in range(N_CLASSES):
            class_totals[c] += (lbl == c).sum()

    all_results.append({
        "frame": fid, "l0": counts[0],
        "l1": counts[1], "l2": counts[2],
        "ms": elapsed,
    })

    print(f"{fid:<10} {counts[0]:>7,} {counts[1]:>7,} "
          f"{counts[2]:>7,} {t_lbl:>7.1f}ms "
          f"{t_vox:>7.1f}ms {t_cell:>7.1f}ms "
          f"{t_save:>7.1f}ms {elapsed:>9.1f}ms "
          f"{1000/elapsed:>5.1f}  {check}")

# ── SUMMARY ───────────────────────────────────────────────
def avg(a): return float(np.mean(a))

print("\n" + "=" * 70)
print("STAGE TIMING (averages)")
print("=" * 70)
print(f"  remap_labels        : {avg(t_lbl_acc):>7.1f} ms")
print(f"  assign_voxel_labels : {avg(t_vox_acc):>7.1f} ms")
print(f"  assign_cell_labels  : {avg(t_cell_acc):>7.1f} ms")
print(f"  save_sparse         : {avg(t_save_acc):>7.1f} ms")
ms_avg = avg([r["ms"] for r in all_results])
print(f"  ─────────────────────────────")
print(f"  Total avg           : {ms_avg:>7.1f} ms")
print(f"  Avg FPS             : {1000/ms_avg:>7.1f}")

print("\n── Class distribution (L1) ──")
total = int(class_totals.sum())
for c in range(N_CLASSES):
    pct = class_totals[c] / max(total,1) * 100
    bar = "█" * int(pct/100*30)
    print(f"  {CLASS_NAMES[c]:<22} "
          f"{class_totals[c]:>7,} ({pct:5.1f}%)  {bar}")

print("\n── sparse5_store contents ──")
npy_files = [
    f for f in os.listdir(SPARSE_DIR)
    if f.endswith(".npy")
]
expected = 10 * 3 * 4
print(f"  .npy files : {len(npy_files)} "
      f"(expected {expected}  = "
      f"10 frames × 3 levels × 4 arrays)")
print(f"  Status     : "
      f"{'✓' if len(npy_files)==expected else '✗'}")
total_kb = sum(
    os.path.getsize(os.path.join(SPARSE_DIR,f))/1024
    for f in npy_files
)
print(f"  Total size : {total_kb:.1f} KB "
      f"= {total_kb/1024:.2f} MB")

# Save metadata
meta_out = {
    "frames":      all_results,
    "n_classes":   N_CLASSES,
    "class_names": CLASS_NAMES,
    "feature_channels": CH,
    "class_distribution_l1": {
        CLASS_NAMES[c]: int(class_totals[c])
        for c in range(N_CLASSES)
    },
}
with open(
    os.path.join(SPARSE_DIR, "dataset_meta.json"), "w"
) as f:
    json.dump(meta_out, f, indent=2)
print(f"\n  Metadata   : {SPARSE_DIR}/dataset_meta.json")

Inverse in voxel files : ✓ fast path

Frame           L0      L1      L2   Lbl ms   Vox ms  Cell ms  Save ms   Total ms    FPS  Check
----------------------------------------------------------------------------------------------------
000000      25,335  13,239   1,567     1.0ms     5.1ms    19.8ms     8.4ms      93.8ms  10.7  ✓
000001      25,268  13,008   1,543     0.7ms     4.1ms    20.1ms     8.0ms      89.8ms  11.1  ✓
000002      25,007  12,651   1,541     0.8ms     4.7ms    38.1ms     8.8ms     111.8ms   8.9  ✓
000003      24,904  12,310   1,498     0.7ms     4.3ms    20.1ms     7.6ms      98.5ms  10.2  ✓
000004      24,710  12,159   1,549     0.7ms     3.6ms    18.3ms     7.7ms      85.8ms  11.7  ✓
000005      24,545  11,982   1,567     0.7ms     4.0ms    17.2ms     7.6ms      85.0ms  11.8  ✓
000006      24,259  11,646   1,495     0.8ms     3.8ms    18.0ms     7.8ms      89.6ms  11.2  ✓
000007      24,112  11,518   1,489     0.7ms     3.8ms    18.0ms     7.4ms      84.3ms  11.9 

In [12]:
!pip install ninja
!pip install -U git+https://github.com/NVIDIA/MinkowskiEngine.git --no-deps -v

  Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 6.8 MB/s eta 0:00:00
Using pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Cloning https://github.com/NVIDIA/MinkowskiEngine.git to /tmp/pip-req-build-k8tul828
  Running command git version
  git version 2.43.0
  Running command git clone --filter=blob:none https://github.com/NVIDIA/MinkowskiEngine.git /tmp/pip-req-build-k8tul828
  Cloning into '/tmp/pip-req-build-k8tul828'...
  Running command git rev-parse HEAD
  02fc608bea4c0549b0a7b00ca1bf15dee4a0b228
  Resolved https://github.com/NVIDIA/MinkowskiEngine.git to commit 02fc608bea4c0549b0a7b00ca1bf15dee4a0b228
  Running command git rev-parse HEAD
  02fc608bea4c0549b0a7b00ca1bf15dee4a0b228
  Running command python setup.py egg_info
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
      exec(compile('''
      ~~~~^^^^^^^^^

In [10]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
